In [0]:
# 0) Cargar la tabla limpia de SILVER
from pyspark.sql import functions as F

clientes = spark.table("workspace.credito_riesgo_lab.clientes_silver")

# .count() dispara el cálculo. Spark es "lazy": no ejecuta nada hasta una acción como count/show/display
print(f"Filas: {clientes.count()}")
clientes.printSchema()

In [0]:
# 1) SELECT + FILTER + ORDER BY — elegir columnas, filtrar filas y ordenar
# Equivalente SQL:
#   SELECT cliente_id, ingresos, endeudamiento, historial_pagos
#   FROM clientes_silver WHERE ingresos > 4000 ORDER BY ingresos DESC
top_ingresos = (
    clientes
    .select("cliente_id", "ingresos", "endeudamiento", "historial_pagos")  # qué columnas quiero
    .filter(F.col("ingresos") > 4000)                                       # qué filas quiero
    .orderBy(F.desc("ingresos"))                                            # orden descendente
)
display(top_ingresos.limit(10))

In [0]:
# 2) GROUP BY + agregaciones — resumir por grupo
# ¿Cuántos clientes hay y cuál es su ingreso/endeudamiento medio según el historial de pagos?
(
    clientes
    .groupBy("historial_pagos")
    .agg(
        F.count("*").alias("clientes"),                   # cuántos hay en cada grupo
        F.round(F.avg("ingresos"), 0).alias("ingreso_medio"),
        F.round(F.avg("endeudamiento"), 2).alias("dti_medio"),
    )
    .orderBy(F.desc("clientes"))
    .show()
)

In [0]:
# 3) Lo mismo con SQL puro
# createOrReplaceTempView pone un "apodo" temporal al DataFrame para consultarlo con SQL
clientes.createOrReplaceTempView("clientes_v")

spark.sql("""
    SELECT historial_pagos,
           COUNT(*)                AS clientes,
           ROUND(AVG(ingresos), 0) AS ingreso_medio
    FROM clientes_v
    GROUP BY historial_pagos
    ORDER BY clientes DESC
""").show()

In [0]:
# 4) Estadísticas descriptivas de las variables numéricas
# summary() devuelve count, mean, stddev, min y max de cada columna elegida
display(clientes.select("ingresos", "endeudamiento", "ltv", "antiguedad_meses", "edad").summary())

# 02 · Exploración básica con PySpark

**Objetivo:** aprender los comandos esenciales para explorar datos: `select`, `filter`, `groupBy`, `orderBy` y su equivalente en SQL. Usamos la tabla limpia **silver**: `workspace.credito_riesgo_lab.clientes_silver`.

> Ejecuta antes `00_datos_sinteticos` y `01_medallion_pipeline`.